# Some very simple classification using scikit-learn
We are going to predict from transactional data whether customers are loyal. We will do this by training a classifier that will predict whether a given customer will return in 2015.

## Load relevant modules

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

## Data Preparation
Get the data from http://liacs.leidenuniv.nl/~takesfw/DSPM/data/sales.csv and read it with Pandas. The argument `names` provide the column headers.

In [ ]:
df = pd.read_csv(
    'http://liacs.leidenuniv.nl/~takesfw/DSPM/data/sales.csv', 
    names=[
        'saleId', 'saleDateTime', 'accountName', 'coins', 'currency', 
        'priceInCurrency', 'priceInEUR', 'methodId', 'ip', 'ipCountry'
    ],
    parse_dates=['saleDateTime']
)
df

## Feature construction
Slice the data such that only transactions before 2015 are included. We want to predict whether customers return in 2015 and hence this information may not be included in the features.


In [ ]:
learning_set = df[df['saleDateTime'].dt.year < 2015]

Create some very basic features:
- number of transactions
- mean priceInCurrency per transactions
- maximum number of coints bought in one transaction

Each row in the following table is one customer (result is similiar to SQL's `GROUP BY` function), each column one feature.

In [ ]:
features = learning_set.groupby(['accountName']).agg({'coins': 'max', 'priceInCurrency': 'mean', 'saleId': 'nunique'})
features

The following array presents all customers that had a transaction in 2015. This are the customers that are loyal, hence we want the classification model to target these customers.

In [ ]:
target = df[df['saleDateTime'].dt.year == 2015]['accountName'].values

In [ ]:
target

In [ ]:
features['target'] = features.index.isin(target)

Count the numbers that did or did not return in 2015. Note the class imbalance. A model predicting always `False` will have an accuracy over 90%.

In [ ]:
features['target'].value_counts()

In [ ]:
train, test = train_test_split(features)

## Clasification

In [ ]:
classifier = RandomForestClassifier()

In [ ]:
classifier.fit(X=train.drop(columns='target'), y=train['target'])

In [ ]:
y_pred = classifier.predict(X=test.drop(columns='target'))

We've trained a model with the following characteristics:

|    | Actual Negatives | Actual Positives |
|:---|:---:|:---:|
| Predicted Negatives | 15373 | 314 |
| Predicted Positives | 1313 | 128 |

In [ ]:
confusion_matrix(y_true=test['target'], y_pred=y_pred)